# Cycle 4 — APIs and Reproducible Acquisition

**Essential question:** How can we request live information responsibly while preserving a result that another person can reproduce?

This canonical teaching notebook is the instructional source of truth for Cycle 4.

## Teaching Notes

Use the fixture path first. Enable a live call only after students can describe the endpoint, parameters, expected schema, timeout, failure path, attribution, and usage limits. Never demonstrate a real secret in a notebook.

## Learning Objectives

Students can explain APIs and HTTP; read endpoint documentation; send a parameterized GET request; distinguish status, headers, and JSON body; recognize authentication patterns; validate a response; handle timeout/HTTP/JSON/schema failures; use a cached fallback; document provenance and respectful use; and normalize a small JSON response.

## Prior Knowledge

Cycles 1–3: functions, dictionaries/lists, exceptions, DataFrames, normalization, validation, provenance, and reproducible paths.

> **Instructor prompt:** Which parts of a live response might change tomorrow, even if our code does not?

## API Intro Classwork - pokeapi.co
Describe at least two endpoints that seem intriguing. What are the route parameters being used in each? What data does it retrieve? Include this as a Markdown block in your notebook. Pull the data from the API as a fixture first and practice importing it into your notebook as a JSON file. If you have time, practice achieving a live connection to the data.

### Endpoint 1: Contest Types

**Endpoint:** `GET https://pokeapi.co/api/v2/contest-type/{id or name}/`

**Route Parameters:** The route parameter is `{id or name}`. It allows the user to get a specific contest type using either its numerical ID (such as `1`, `2`, etc.) or its name (e.g. `cool`, `beautiful`, etc.)

**Data Retrieved:** This endpoint returns information about a Pokémon contest type, including its ID, name, associated berry flavor, and names in differnet langauges. One example is the "cool" contest type has a corresponding ID of `1`, and is associated with the "spicy" berry flavor, and is associated with the color "red". 

### Endpoint 2: Currencies

**Endpoint:** `GET https://pokeapi.co/api/v2/currency/{id or name}/`

**Route Parameters:** The route parameter is `{id or name}`. It allows the user to get a specific currency using either its numerical ID (e.g., `1`) or its name (e.g., `poke-dollar`).

**Data Retrieved:** This endpoint returns information about a currency used in Pokémon games, including its ID, name, and names in different languages. For example, the "poke-dollar" currency has an ID of `1` and is called "Poké Dollar" in English.

In [12]:
import json

# Load contest type data
with open("contest_type.json", encoding="utf-8") as handle:
    contest_data = json.load(handle)

print(contest_data.keys())

# Load currency data
with open("currency.json", encoding="utf-8") as handle:
    currency_data = json.load(handle)

print(currency_data.keys())

dict_keys(['id', 'name', 'berry_flavor', 'names'])
dict_keys(['id', 'name', 'names'])


In [13]:
# Contest type
print("Contest Type:", contest_data["name"])
print("ID:", contest_data["id"])
print("Berry Flavor:", contest_data["berry_flavor"]["name"])

# Currency
print("Currency:", currency_data["name"])
print("ID:", currency_data["id"])

Contest Type: tough
ID: 5
Berry Flavor: sour
Currency: poke-dollar
ID: 1


In [ ]:
import requests

# Contest 
CONTEST_ENDPOINT = "https://pokeapi.co/api/v2/contest-type/1/"

contest_response = requests.get(CONTEST_ENDPOINT)
contest_response.raise_for_status()

live_contest_data = contest_response.json()

print(live_contest_data.keys())
print("Contest Type:", live_contest_data["name"])
print("Berry Flavor:", live_contest_data["berry_flavor"]["name"])

# Currency 
CURRENCY_ENDPOINT = "https://pokeapi.co/api/v2/currency/1/"
currency_response = requests.get(CURRENCY_ENDPOINT)
currency_response.raise_for_status()

live_currency_data = currency_response.json()

print(live_currency_data.keys())
print("Currency:", live_currency_data["name"])
print("ID:", live_currency_data["id"])

dict_keys(['id', 'name', 'berry_flavor', 'names'])
Contest Type: cool
Berry Flavor: spicy
dict_keys(['id', 'name', 'names'])
Currency: poke-dollar
ID: 1


## 1. API and HTTP Model

An API is a documented interface. A GET request combines a base endpoint, parameters, and headers. A response has a status code, headers, and body. JSON represents objects as dictionaries and arrays as lists.

In [15]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path("./")
FIXTURE_DIR = DATA_DIR / "api-fixtures"
assert FIXTURE_DIR.exists(), "Run from the teaching-notebook directory and keep datasets beside it."

In [16]:
fixture_path = FIXTURE_DIR / "open-meteo-nyc-forecast.json"
with fixture_path.open(encoding="utf-8") as handle:
    fixture = json.load(handle)
print(fixture.keys())
print(fixture["daily"].keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum'])


## 2. Read Documentation Before Requesting

For Open-Meteo `/v1/forecast`, latitude and longitude are required. We request only three daily variables, seven days, and a named timezone. Free classroom use needs no key; data attribution is required. The fixture registry records limits and retrieval details.

In [17]:
ENDPOINT = "https://api.open-meteo.com/v1/forecast" # ENDPOINT
PARAMS = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/New_York",
    "forecast_days": 7,
}
EXPECTED_DAILY = {"time", "temperature_2m_max", "temperature_2m_min", "precipitation_sum"}

## 3. Authentication Concepts

APIs may use no credential, an API key, OAuth, or another token. Read secrets from environment variables such as `os.getenv("SERVICE_API_KEY")`; never print or commit them. A missing credential should produce a clear message and use an approved fixture when possible.

## 4. Validated Live-or-Fixture Acquisition

The default is offline. A teacher may set `use_live=True`. The function uses a timeout, checks HTTP status, parses JSON, validates schema, and falls back after any request or validation failure.

In [18]:
def validate_weather_payload(payload):
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")
    if "daily" not in payload or not isinstance(payload["daily"], dict):
        raise ValueError("Missing daily object")
    missing = EXPECTED_DAILY - set(payload["daily"])
    if missing:
        raise ValueError(f"Missing daily fields: {sorted(missing)}")
    lengths = {key: len(payload["daily"][key]) for key in EXPECTED_DAILY}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"Daily arrays have different lengths: {lengths}")
    return payload

validate_weather_payload(fixture)

{'latitude': 40.710335,
 'longitude': -73.99308,
 'generationtime_ms': 0.07176399230957031,
 'utc_offset_seconds': -14400,
 'timezone': 'America/New_York',
 'timezone_abbreviation': 'GMT-4',
 'elevation': 32.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2026-07-30',
   '2026-07-31',
   '2026-08-01',
   '2026-08-02',
   '2026-08-03',
   '2026-08-04',
   '2026-08-05'],
  'temperature_2m_max': [23.7, 30.1, 29.7, 27.4, 29.8, 25.6, 25.6],
  'temperature_2m_min': [19.5, 18.7, 20.9, 23.6, 23.2, 23.7, 23.5],
  'precipitation_sum': [0.0, 0.0, 0.0, 0.0, 17.2, 8.4, 11.4]}}

In [19]:
def acquire_weather(use_live=False):
    source = "cached fixture"
    payload = fixture
    if use_live:
        try:
            import requests
            response = requests.get(ENDPOINT, params=PARAMS, timeout=10)
            response.raise_for_status()
            payload = response.json()
            validate_weather_payload(payload)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = fixture
    return validate_weather_payload(payload), source

weather_payload, acquisition_source = acquire_weather(use_live=False)
acquisition_source

'cached fixture'

### Intentional Bug

This example omits timeout, status checking, schema validation, and fallback. It is a pattern to critique, not copy.

In [20]:
# Fragile pattern: do not use for submitted work.
import requests
requests.get(ENDPOINT, params=PARAMS).json()

{'latitude': 40.710335,
 'longitude': -73.99308,
 'generationtime_ms': 0.1100301742553711,
 'utc_offset_seconds': -18000,
 'timezone': 'America/New_York',
 'timezone_abbreviation': 'GMT-5',
 'elevation': 32.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2026-11-09',
   '2026-11-10',
   '2026-11-11',
   '2026-11-12',
   '2026-11-13',
   '2026-11-14',
   '2026-11-15'],
  'temperature_2m_max': [15.1, 16.5, 15.3, 15.7, 11.3, 6.6, 8.5],
  'temperature_2m_min': [7.4, 5.5, 5.1, 11.3, 4.5, 4.1, 4.0],
  'precipitation_sum': [0.0, 0.0, 0.0, 3.1, 2.3, 0.0, 0.0]}}

## 5. Normalize and Record Provenance

In [21]:
daily = pd.DataFrame(weather_payload["daily"])
daily["time"] = pd.to_datetime(daily["time"])
assert len(daily) == 7
assert EXPECTED_DAILY <= set(daily.columns)
daily

,time,temperature_2m_max,temperature_2m_min,precipitation_sum
0,2026-07-30,23.7,19.5,0.0
1,2026-07-31,30.1,18.7,0.0
2,2026-08-01,29.7,20.9,0.0
3,2026-08-02,27.4,23.6,0.0
4,2026-08-03,29.8,23.2,17.2
5,2026-08-04,25.6,23.7,8.4
6,2026-08-05,25.6,23.5,11.4


In [22]:
provenance = {
    "provider": "Open-Meteo",
    "endpoint": ENDPOINT,
    "parameters": PARAMS,
    "acquisition_source": acquisition_source,
    "fixture_retrieved": "2026-07-30",
    "license": "CC BY 4.0; attribution required",
}
provenance

{'provider': 'Open-Meteo',
 'endpoint': 'https://api.open-meteo.com/v1/forecast',
 'parameters': {'latitude': 40.7128,
  'longitude': -74.006,
  'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum',
  'timezone': 'America/New_York',
  'forecast_days': 7},
 'acquisition_source': 'cached fixture',
 'fixture_retrieved': '2026-07-30',
 'license': 'CC BY 4.0; attribution required'}

## Guided Practice

Trace endpoint → parameters → status → JSON → validation → DataFrame. Simulate a missing `daily` key and explain where the function fails. Identify the attribution and usage rule in the fixture registry.

## Independent Practice

Build an offline-first acquisition notebook containing documentation notes, parameter dictionary, timeout, failure handling, fixture fallback, schema checks, normalized DataFrame, provenance record, one exploratory observation, and reflection.

## Challenge

Add a `required_fields` parameter to the validator or support an optional environment-based API key without revealing it.

## Reflection

What changes between live and archived data? Which check prevents the most misleading failure? How does caching support respectful use and reproducibility?